In [ ]:
#Importamos las librerías que vamos a utilizar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
ruta = r"C:/Users\Asus/OneDrive - frsr.utn.edu.ar/Visual Studio/PythonProyecto/YPF DS - Proyecto Final/Material/Datasets/ai4i2020.csv"
df = pd.read_csv(ruta)

In [25]:
# Mostramos la cantidad de filas y columnas del DataFrame
print("El DataFrame tiene", df.shape[0], "filas y", df.shape[1] ,"columnas")

El DataFrame tiene 10000 filas y 14 columnas


In [24]:
# Mostramos las primeras filas del DataFrame
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [36]:
# Creamos un nuevo DataFrame para juntar informaciones
info_df = pd.DataFrame({
    'Tipo de Dato': df.dtypes,
    'Valores Nulos': df.isnull().sum(),
})

# Asignamos un nombre a la primera columna del DataFrame
info_df = info_df.reset_index().rename(columns={'index': 'Variables'})

info_df

,Variables,Tipo de Dato,Valores Nulos
0,UDI,int64,0
1,Product ID,object,0
2,Type,object,0
3,Air temperature [K],float64,0
4,Process temperature [K],float64,0
5,Rotational speed [rpm],int64,0
6,Torque [Nm],float64,0
7,Tool wear [min],int64,0
8,Machine failure,int64,0
9,TWF,int64,0


In [45]:
# Generamos las estadísticas descriptivas del DataFrame
df.describe()

Variables,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2886.89568,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,1.00000,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,5000.50000,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7500.25000,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,10000.00000,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


### 🔍 Diagnóstico del Balance de Clases 


In [ ]:
# Vamos a ver la cantidad de fallas que hay en el dataset
conteo_fallas = df['Machine failure'].value_counts()

# Calculamos el porcentaje de fallas en el dataset
porcentaje_fallas = df['Machine failure'].value_counts(normalize=True) * 100

# Resumen 
resumen_fallas = pd.DataFrame({
    'Cantidad de fallas' : conteo_fallas,
    'Porcentaje (%)' : porcentaje_fallas.round(4)
})

resumen_fallas.index.name = None

resumen_fallas

,Cantidad de fallas,Porcentaje (%)
0,9661,96.61
1,339,3.39


A partir del análisis cuantitativo de la variable objetivo `Machine failure`, se extraen las siguientes conclusiones analíticas fundamentales para el desarrollo del modelo:

* **Predominancia de la Clase Mayoritaria:** El **96.61%** de los registros históricos corresponden a un estado operativo normal (Clase `0`), sumando un total de **9,661 observaciones**.
* **Presencia de la Clase Minoritaria (Target):** Se registraron únicamente **339 eventos de falla mecánica** (Clase `1`), lo que equivale al **3.39%** del conjunto de datos total.

### 🛠️ Análisis Desagregado de Tipos de Falla


In [ ]:
# Creamos una lista con las columnas de fallas específicas
columnas_fallas = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# Sumamos los valores de cada columna
conteo_fallas_especificas = df[columnas_fallas].sum()

# Creamos un DataFrame y lo ordenamos de mayor a menor
tabla_fallas_especificas = pd.DataFrame({
    'Total de Eventos': conteo_fallas_especificas,
}).sort_values(by='Total de Eventos', ascending=False)

# Quitamos el nombre del índice para que quede más limpio
tabla_fallas_especificas.index.name = None

tabla_fallas_especificas

,Total de Eventos
HDF,115
OSF,98
PWF,95
TWF,46
RNF,19


In [79]:
# Contabilizamos el total de fallos de todas las fallas específicas
print("Tenemos un total de:", conteo_fallas_especificas.sum(), "fallas teniendo en cuenta todas las fallas específicas")

Tenemos un total de: 373 fallas teniendo en cuenta todas las fallas específicas


Para entender la naturaleza del desbalance, se aislaron y contabilizaron de forma independiente los modos de falla registrados por los sensores mecánicos:

* **HDF (Heat Dissipation Failure - 115 eventos):** Es la causa principal de avería. Indica problemas críticos de disipación térmica y sobrecalentamiento en la operación.
* **OSF (Overstrain Failure - 98 eventos):** El segundo factor de riesgo, asociado a tensiones mecánicas excesivas o sobreesfuerzos estructurales.
* **PWF (Power Failure - 95 eventos):** Fallos directamente relacionados con fluctuaciones o pérdidas en el suministro de potencia eléctrica.
* **TWF (Tool Wear Failure - 45 eventos):** Desgaste acumulado de la herramienta de corte que alcanza su límite seguro.
* **RNF (Random Failure - 18 eventos):** Eventos de fallo aleatorios no correlacionados directamente con los sensores principales.
<br>
<br>
> 💡 **Nota de Consistencia de Datos:** Si sumamos todos estos eventos individuales obtenemos 373 fallas, lo cual supera las 339 fallas registradas identificadas con 'Machine failure = 1'. Este hallazgo es clave para nuestro análisis EDA: **existen ciclos de operación específicos donde la máquina sufrió múltiples modos de falla simultáneos** durante el mecanizado de una pieza.



In [ ]:
# Totales de fallas
total_fallas = df['Machine failure'].sum()
suma_fallas_especificas = df[columnas_fallas].sum().sum()

# Contamos los ciclos que tuvieron más de una falla a la vez
num_fallas_ciclos = (df[columnas_fallas].sum(axis=1) >= 2).sum()

#
tabla_fallas = pd.DataFrame({
    'Metrica Analizada': [
        'Total Alarmas Generales (Machine failure = 1)', 
                'Suma Total de Fallas Específicas Detectadas',
                'Diferencia Bruta en el Conteo',
                'Cantidad de Ciclos con Múltiples Fallas Simultáneas'
    ],
    'Valor':[
        total_fallas, 
        suma_fallas_especificas, 
        suma_fallas_especificas - total_fallas,
        num_fallas_ciclos
    ]
})

tabla_fallas

#
((df['Machine failure'] == 0) & (df[columnas_fallas])>1).sum()

In [115]:
# 1. Definimos las columnas de fallas específicas
columnas_fallas = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# 2. Obtenemos los totales absolutos
total_machine_failure = df['Machine failure'].sum()
suma_fallas_especificas = df[columnas_fallas].sum().sum()

# 3. Contamos los ciclos que tuvieron exactamente más de una falla a la vez
ciclos_multi_falla = (df[columnas_fallas].sum(axis=1) >= 2).sum()

# 4. Creamos una tabla resumen para demostrar la diferencia
tabla_demostración = pd.DataFrame({
    'Métrica Analizada': [
        'Total Alarmas Generales (Machine failure = 1)', 
        'Suma Total de Fallas Específicas Detectadas',
        'Diferencia Bruta en el Conteo',
        'Cantidad de Ciclos con Múltiples Fallas Simultáneas'
    ],
    'Valor': [
        total_machine_failure, 
        suma_fallas_especificas, 
        suma_fallas_especificas - total_machine_failure,
        ciclos_multi_falla
    ]
})

tabla_demostración

,Métrica Analizada,Valor
0,Total Alarmas Generales (Machine failure = 1),339
1,Suma Total de Fallas Específicas Detectadas,373
2,Diferencia Bruta en el Conteo,34
3,Cantidad de Ciclos con Múltiples Fallas Simult...,24


In [24]:
#  Análisis de desbalance de clases (Machine failure y fallas específicas)
for col in ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']:
    print(col)
    print(df[col].value_counts(normalize=True) * 100)
    print("---")

for col in fail_cols:
    n = df[col].sum()
    pct = df[col].mean() * 100
    print(f"{col}: {n} casos ({pct:.2f}%)")

Machine failure
Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64
---
TWF
TWF
0    99.54
1     0.46
Name: proportion, dtype: float64
---
HDF
HDF
0    98.85
1     1.15
Name: proportion, dtype: float64
---
PWF
PWF
0    99.05
1     0.95
Name: proportion, dtype: float64
---
OSF
OSF
0    99.02
1     0.98
Name: proportion, dtype: float64
---
RNF
RNF
0    99.81
1     0.19
Name: proportion, dtype: float64
---
Machine failure: 339 casos (3.39%)
TWF: 46 casos (0.46%)
HDF: 115 casos (1.15%)
PWF: 95 casos (0.95%)
OSF: 98 casos (0.98%)
RNF: 19 casos (0.19%)
